# 01. 데이터 로드 및 파일 점검
목표는 아직 정제나 분석을 수행하는 것이 아니라, 원본 파일이 정상적으로 준비되었는지 확인하는 것입니다.

확인 항목은 다음과 같습니다.

1. 파일이 정상적으로 읽히는가  
2. 한글이 깨지지 않는가  
3. 컬럼명이 예상과 일치하는가  
4. 각 파일의 행 수가 정상적인가  
5. 분석 기간 `2024.01 ~ 2025.12`가 포함되어 있는가  
6. 대여소 정보 XLSX에서 올바른 시트를 읽을 수 있는가  


## 0. 라이브러리 불러오기

In [20]:
from pathlib import Path
from datetime import datetime
import re

import pandas as pd


## 1. 경로 설정

현재 프로젝트에서는 아래 구조를 기준으로 합니다.

```text
seoul-bike-aarrr-analysis/
├─ data/
│  └─ raw/
│     ├─ signup_month/
│     ├─ ride_month/
│     ├─ station_snapshot/
│     └─ station_usage_month/
└─ notebooks/
   └─ 01_data_load_and_check.ipynb
```

현재 노트북이 `notebooks/` 폴더 안에서 실행된다고 가정합니다.  
단, 실습 환경에서 `/mnt/data`에 파일이 바로 있는 경우도 자동으로 찾도록 보조 로직을 넣었습니다.


In [2]:
# 현재 노트북 위치 기준 프로젝트 루트 추정
cwd = Path.cwd()

if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

RAW_DIR = PROJECT_ROOT / "data" / "raw"

# ChatGPT 업로드 환경처럼 /mnt/data에 파일이 바로 있는 경우를 위한 fallback
if not RAW_DIR.exists() and Path("/mnt/data").exists():
    RAW_DIR = Path("/mnt/data")

print("현재 작업 폴더:", cwd)
print("프로젝트 루트:", PROJECT_ROOT)
print("원본 데이터 폴더:", RAW_DIR)


현재 작업 폴더: c:\Users\위동국\Documents\GitHub\seoul-bike-aarrr-analysis\notebooks
프로젝트 루트: c:\Users\위동국\Documents\GitHub\seoul-bike-aarrr-analysis
원본 데이터 폴더: c:\Users\위동국\Documents\GitHub\seoul-bike-aarrr-analysis\data\raw


## 2. 파일 목록 수집

로컬 프로젝트 구조에 맞게 파일을 폴더별로 정리했다면 각 하위 폴더에서 파일을 찾습니다.  
아직 한 폴더에 파일이 모두 섞여 있다면 파일명 패턴으로도 찾을 수 있습니다.


In [3]:
def find_files(folder_name: str, pattern: str):
    folder = RAW_DIR / folder_name
    if folder.exists():
        found = sorted(folder.glob(pattern))
    else:
        found = sorted(RAW_DIR.glob(pattern))
    return found

files = {
    "signup_month": find_files("signup_month", "서울특별시 공공자전거 신규가입자 정보(월별)_*.csv"),
    "ride_month": find_files("ride_month", "서울특별시 공공자전거 이용정보(월별)_*.csv"),
    "station_snapshot": find_files("station_snapshot", "공공자전거 대여소 정보(*기준).xlsx"),
    "station_usage_month": find_files("station_usage_month", "서울특별시 공공자전거 대여소별 이용정보(월별)_*.csv"),
}

for key, paths in files.items():
    print(f"[{key}] {len(paths)}개")
    for p in paths:
        print(" -", p.name)


[signup_month] 4개
 - 서울특별시 공공자전거 신규가입자 정보(월별)_24.1-6.csv
 - 서울특별시 공공자전거 신규가입자 정보(월별)_24.7-12.csv
 - 서울특별시 공공자전거 신규가입자 정보(월별)_25.1-6.csv
 - 서울특별시 공공자전거 신규가입자 정보(월별)_25.7-12.csv
[ride_month] 4개
 - 서울특별시 공공자전거 이용정보(월별)_24.1-6.csv
 - 서울특별시 공공자전거 이용정보(월별)_24.7-12.csv
 - 서울특별시 공공자전거 이용정보(월별)_25.1-6.csv
 - 서울특별시 공공자전거 이용정보(월별)_25.7-12.csv
[station_snapshot] 4개
 - 공공자전거 대여소 정보(24.12월 기준).xlsx
 - 공공자전거 대여소 정보(24.6월 기준).xlsx
 - 공공자전거 대여소 정보(25.12월 기준).xlsx
 - 공공자전거 대여소 정보(25.6월 기준).xlsx
[station_usage_month] 4개
 - 서울특별시 공공자전거 대여소별 이용정보(월별)_24.1-6.csv
 - 서울특별시 공공자전거 대여소별 이용정보(월별)_24.7-12.csv
 - 서울특별시 공공자전거 대여소별 이용정보(월별)_25.1-6.csv
 - 서울특별시 공공자전거 대여소별 이용정보(월별)_25.7-12.csv


## 3. CSV 인코딩 및 기본 구조 확인

공공데이터 CSV는 `cp949`, `utf-8-sig` 등 인코딩이 섞여 있을 수 있습니다.  
따라서 여러 인코딩을 순서대로 시도해 정상적으로 읽히는 방식을 찾습니다.


In [4]:
encoding_candidates = ["utf-8-sig", "utf-8", "cp949", "euc-kr"]

def detect_csv_encoding(path: Path):
    for enc in encoding_candidates:
        try:
            pd.read_csv(path, encoding=enc, nrows=5)
            return enc
        except Exception:
            continue
    return None

def csv_basic_check(path: Path, dataset_key: str, period_col: str | None):
    enc = detect_csv_encoding(path)

    if enc is None:
        return {
            "dataset": dataset_key,
            "file_name": path.name,
            "encoding": "READ_FAILED",
            "row_count": None,
            "columns": None,
            "period_min": None,
            "period_max": None,
            "status": "fail: cannot detect encoding"
        }

    try:
        header = pd.read_csv(path, encoding=enc, nrows=0)
        columns = list(header.columns)

        row_count = 0
        period_min, period_max = None, None

        usecols = [period_col] if period_col and period_col in columns else None

        for chunk in pd.read_csv(path, encoding=enc, usecols=usecols, chunksize=200_000):
            row_count += len(chunk)

            if usecols:
                vals = pd.to_numeric(chunk[period_col], errors="coerce").dropna()
                if len(vals) > 0:
                    cmin, cmax = int(vals.min()), int(vals.max())
                    period_min = cmin if period_min is None else min(period_min, cmin)
                    period_max = cmax if period_max is None else max(period_max, cmax)

        return {
            "dataset": dataset_key,
            "file_name": path.name,
            "encoding": enc,
            "row_count": row_count,
            "columns": ", ".join(columns),
            "period_min": period_min,
            "period_max": period_max,
            "status": "ok"
        }

    except Exception as e:
        return {
            "dataset": dataset_key,
            "file_name": path.name,
            "encoding": enc,
            "row_count": None,
            "columns": None,
            "period_min": None,
            "period_max": None,
            "status": f"fail: {type(e).__name__}: {e}"
        }


In [5]:
csv_checks = []

for p in files["signup_month"]:
    csv_checks.append(csv_basic_check(p, "신규가입자 정보(월별)", "가입년월"))

for p in files["ride_month"]:
    csv_checks.append(csv_basic_check(p, "이용정보(월별)", "대여일자"))

for p in files["station_usage_month"]:
    csv_checks.append(csv_basic_check(p, "대여소별 이용정보(월별)", "기준년월"))

csv_check_df = pd.DataFrame(csv_checks)
csv_check_df


,dataset,file_name,encoding,row_count,columns,period_min,period_max,status
0,신규가입자 정보(월별),서울특별시 공공자전거 신규가입자 정보(월별)_24.1-6.csv,cp949,96,"가입년월, 회원구분, 연령대, 성별, 가입건수",202401,202406,ok
1,신규가입자 정보(월별),서울특별시 공공자전거 신규가입자 정보(월별)_24.7-12.csv,cp949,96,"가입년월, 회원구분, 연령대, 성별, 가입건수",202407,202412,ok
2,신규가입자 정보(월별),서울특별시 공공자전거 신규가입자 정보(월별)_25.1-6.csv,cp949,97,"가입년월, 회원구분, 연령대, 성별, 가입건수",202501,202506,ok
3,신규가입자 정보(월별),서울특별시 공공자전거 신규가입자 정보(월별)_25.7-12.csv,cp949,96,"가입년월, 회원구분, 연령대, 성별, 가입건수",202507,202512,ok
4,이용정보(월별),서울특별시 공공자전거 이용정보(월별)_24.1-6.csv,cp949,613732,"대여일자, 대여소번호, 대여소명, 대여구분코드, 성별, 연령대코드, 이용건수, 운동...",202401,202406,ok
5,이용정보(월별),서울특별시 공공자전거 이용정보(월별)_24.7-12.csv,cp949,619664,"대여일자, 대여소번호, 대여소명, 대여구분코드, 성별, 연령대코드, 이용건수, 운동...",202407,202412,ok
6,이용정보(월별),서울특별시 공공자전거 이용정보(월별)_25.1-6.csv,cp949,600437,"대여일자, 대여소번호, 대여소명, 대여구분코드, 성별, 연령대코드, 이용건수, 운동...",202501,202506,ok
7,이용정보(월별),서울특별시 공공자전거 이용정보(월별)_25.7-12.csv,cp949,629975,"대여일자, 대여소번호, 대여소명, 대여구분코드, 성별, 연령대코드, 이용건수, 운동...",202507,202512,ok
8,대여소별 이용정보(월별),서울특별시 공공자전거 대여소별 이용정보(월별)_24.1-6.csv,cp949,16382,"자치구, 대여소명, 기준년월, 대여건수, 반납건수",202401,202406,ok
9,대여소별 이용정보(월별),서울특별시 공공자전거 대여소별 이용정보(월별)_24.7-12.csv,cp949,16404,"자치구, 대여소명, 기준년월, 대여건수, 반납건수",202407,202412,ok


## 4. XLSX 대여소 정보 시트 확인

대여소 정보 파일은 CSV가 아니라 XLSX입니다.  
따라서 시트 목록을 확인하고, 실제 데이터가 들어 있는 `대여소현황` 시트를 읽을 수 있는지 확인합니다.

또한 대여소 정보 파일에는 기준월 컬럼이 없으므로, 파일명에서 `snapshot_ym`을 추출합니다.


In [16]:
def extract_snapshot_ym(file_name: str):
    # 예: 공공자전거 대여소 정보(24.6월 기준).xlsx
    m = re.search(r"\((\d{2})\.(\d{1,2})월 기준\)", file_name)
    if not m:
        return None
    yy, mm = m.groups()
    return int(f"20{yy}{int(mm):02d}")

def find_station_header_row(xlsx_path: Path, sheet_name="대여소현황", max_rows=20):
    preview = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=None, nrows=max_rows)
    header_keywords = ["대여소", "보관소", "자치구", "위도", "경도"]

    best_idx, best_score = None, -1
    for idx, row in preview.iterrows():
        values = " ".join([str(x) for x in row.values if pd.notna(x)])
        score = sum(kw in values for kw in header_keywords)
        if score > best_score:
            best_idx, best_score = idx, score

    return best_idx

def station_xlsx_check(path: Path):
    try:
        xl = pd.ExcelFile(path)
        sheets = xl.sheet_names
        target_sheet = "대여소현황" if "대여소현황" in sheets else sheets[0]

        header_row = find_station_header_row(path, target_sheet)

        df = pd.read_excel(path, sheet_name=target_sheet, header=header_row)
        df = df.dropna(how="all")
        df.columns = [str(c).strip().replace("\n", " ") for c in df.columns]

        first_col = df.columns[0]
        data_rows = df[df[first_col].notna()].copy()

        return {
            "dataset": "대여소 정보",
            "file_name": path.name,
            "snapshot_ym": extract_snapshot_ym(path.name),
            "sheet_names": ", ".join(sheets),
            "used_sheet": target_sheet,
            "header_row_0_based": header_row,
            "row_count_after_header": len(data_rows),
            "columns": ", ".join(df.columns),
            "status": "ok"
        }

    except Exception as e:
        return {
            "dataset": "대여소 정보",
            "file_name": path.name,
            "snapshot_ym": extract_snapshot_ym(path.name),
            "sheet_names": None,
            "used_sheet": None,
            "header_row_0_based": None,
            "row_count_after_header": None,
            "columns": None,
            "status": f"fail: {type(e).__name__}: {e}"
        }


In [17]:
xlsx_checks = [station_xlsx_check(p) for p in files["station_snapshot"]]
xlsx_check_df = pd.DataFrame(xlsx_checks)
xlsx_check_df


,dataset,file_name,snapshot_ym,sheet_names,used_sheet,header_row_0_based,row_count_after_header,columns,status
0,대여소 정보,공공자전거 대여소 정보(24.12월 기준).xlsx,202412,대여소현황,대여소현황,2,2766,"Unnamed: 0, Unnamed: 1, 자치구, 상세주소, 위도, 경도, Unn...",ok
1,대여소 정보,공공자전거 대여소 정보(24.6월 기준).xlsx,202406,대여소현황,대여소현황,2,2763,"Unnamed: 0, Unnamed: 1, 자치구, 상세주소, 위도, 경도, Unn...",ok
2,대여소 정보,공공자전거 대여소 정보(25.12월 기준).xlsx,202512,대여소현황,대여소현황,2,2799,"Unnamed: 0, Unnamed: 1, 자치구, 상세주소, 위도, 경도, Unn...",ok
3,대여소 정보,공공자전거 대여소 정보(25.6월 기준).xlsx,202506,대여소현황,대여소현황,2,2780,"Unnamed: 0, Unnamed: 1, 자치구, 상세주소, 위도, 경도, Unn...",ok


## 5. 데이터셋별 요약

파일 수, 총 행 수, 기간 범위, 정상 읽기 여부를 데이터셋별로 요약합니다.


In [8]:
dataset_summary_rows = []

if not csv_check_df.empty:
    for dataset in csv_check_df["dataset"].unique():
        sub = csv_check_df[csv_check_df["dataset"] == dataset]
        dataset_summary_rows.append({
            "dataset": dataset,
            "file_count": len(sub),
            "total_rows": int(sub["row_count"].fillna(0).sum()),
            "period_min": int(sub["period_min"].min()) if sub["period_min"].notna().any() else None,
            "period_max": int(sub["period_max"].max()) if sub["period_max"].notna().any() else None,
            "all_status_ok": bool((sub["status"] == "ok").all())
        })

if not xlsx_check_df.empty:
    dataset_summary_rows.append({
        "dataset": "대여소 정보",
        "file_count": len(xlsx_check_df),
        "total_rows": int(xlsx_check_df["row_count_after_header"].fillna(0).sum()),
        "period_min": int(xlsx_check_df["snapshot_ym"].min()) if xlsx_check_df["snapshot_ym"].notna().any() else None,
        "period_max": int(xlsx_check_df["snapshot_ym"].max()) if xlsx_check_df["snapshot_ym"].notna().any() else None,
        "all_status_ok": bool((xlsx_check_df["status"] == "ok").all())
    })

dataset_summary_df = pd.DataFrame(dataset_summary_rows)
dataset_summary_df


,dataset,file_count,total_rows,period_min,period_max,all_status_ok
0,신규가입자 정보(월별),4,385,202401,202512,True
1,이용정보(월별),4,2463808,202401,202512,True
2,대여소별 이용정보(월별),4,65761,202401,202512,True
3,대여소 정보,4,0,202406,202512,False


## 6. 한글 깨짐 및 샘플 데이터 확인

각 데이터셋별 첫 번째 파일을 일부 읽어 컬럼명과 샘플 값을 확인합니다.  
한글이 깨져 보이지 않으면 인코딩은 정상으로 판단할 수 있습니다.


In [18]:
preview_rows = []

for key in ["signup_month", "ride_month", "station_usage_month"]:
    if files[key]:
        path = files[key][0]
        enc = detect_csv_encoding(path)
        try:
            df = pd.read_csv(path, encoding=enc, nrows=3)
            preview_rows.append({
                "dataset_key": key,
                "file_name": path.name,
                "encoding": enc,
                "columns": ", ".join(df.columns.astype(str).tolist()),
                "sample_json": df.head(2).to_json(force_ascii=False, orient="records")
            })
        except Exception as e:
            preview_rows.append({
                "dataset_key": key,
                "file_name": path.name,
                "encoding": enc,
                "columns": None,
                "sample_json": f"failed: {e}"
            })

if files["station_snapshot"]:
    path = files["station_snapshot"][0]
    try:
        header_row = find_station_header_row(path, "대여소현황")
        df = pd.read_excel(path, sheet_name="대여소현황", header=header_row, nrows=3)
        preview_rows.append({
            "dataset_key": "station_snapshot",
            "file_name": path.name,
            "encoding": "xlsx",
            "columns": ", ".join([str(c).strip() for c in df.columns]),
            "sample_json": df.head(2).to_json(force_ascii=False, orient="records")
        })
    except Exception as e:
        preview_rows.append({
            "dataset_key": "station_snapshot",
            "file_name": path.name,
            "encoding": "xlsx",
            "columns": None,
            "sample_json": f"failed: {e}"
        })

preview_df = pd.DataFrame(preview_rows)
preview_df


,dataset_key,file_name,encoding,columns,sample_json
0,signup_month,서울특별시 공공자전거 신규가입자 정보(월별)_24.1-6.csv,cp949,"가입년월, 회원구분, 연령대, 성별, 가입건수","[{""가입년월"":202401,""회원구분"":""회원-내국인"",""연령대"":""~10대"",""..."
1,ride_month,서울특별시 공공자전거 이용정보(월별)_24.1-6.csv,cp949,"대여일자, 대여소번호, 대여소명, 대여구분코드, 성별, 연령대코드, 이용건수, 운동...","[{""대여일자"":202401,""대여소번호"":102,""대여소명"":""102. 망원역 1..."
2,station_usage_month,서울특별시 공공자전거 대여소별 이용정보(월별)_24.1-6.csv,cp949,"자치구, 대여소명, 기준년월, 대여건수, 반납건수","[{""자치구"":""강남구"",""대여소명"":""2301. 현대고등학교 건너편"",""기준년월""..."
3,station_snapshot,공공자전거 대여소 정보(24.12월 기준).xlsx,xlsx,"Unnamed: 0, Unnamed: 1, 자치구, 상세주소, 위도, 경도, Unn...","[{""Unnamed: 0"":null,""Unnamed: 1"":null,""자치구"":nu..."


## 7. 점검 결과 저장

점검 결과를 `reports/` 폴더에 저장합니다.  
`reports/` 폴더가 없으면 자동으로 생성합니다.


In [19]:
REPORT_DIR = PROJECT_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

summary_md_path = REPORT_DIR / "data_check_summary.md"

def df_to_md(df):
    return df.to_markdown(index=False)

md = f"""# 01 데이터 로드 및 파일 점검 요약

생성 시각: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## 점검 목적

이 문서는 `notebooks/01_data_load_and_check.ipynb`에서 확인해야 할 내용을 기준으로, 현재 원본 파일이 정상적으로 읽히는지 점검한 결과이다.

확인 항목은 다음과 같다.

1. 파일이 정상적으로 읽히는가
2. 한글이 깨지지 않는가
3. 컬럼명이 예상과 일치하는가
4. 각 파일의 행 수가 정상적인가
5. 분석 기간 2024.01~2025.12가 포함되어 있는가
6. 대여소 정보 XLSX에서 올바른 시트를 읽을 수 있는가

## 데이터셋 요약

{df_to_md(dataset_summary_df)}

## CSV 파일별 점검 결과

{df_to_md(csv_check_df)}

## 대여소 정보 XLSX 점검 결과

{df_to_md(xlsx_check_df)}

"""

summary_md_path.write_text(md, encoding="utf-8")

print("저장 완료")
print(summary_md_path)


저장 완료
c:\Users\위동국\Documents\GitHub\seoul-bike-aarrr-analysis\reports\data_check_summary.md


## 8. 완료 판단

아래 조건을 만족하면 완료로 판단합니다.

- 필수 데이터셋 4종의 파일이 모두 존재한다.
- CSV 파일이 정상적으로 읽힌다.
- 한글 컬럼명과 샘플 값이 깨지지 않는다.
- 신규가입자/이용정보/대여소별 이용정보의 기간이 `2024.01 ~ 2025.12`를 포함한다.
- 대여소 정보 XLSX에서 `대여소현황` 시트를 읽을 수 있다.
- 대여소 정보 파일명에서 `snapshot_ym`을 추출할 수 있다.

